## **Web assistant via `RAG` Powered**

- llamaParser for reading documents
- hydride search system
- advanced re-ranker algo
- self corrective powered rag
- zero hallucinations

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Load Environment Variables
load_dotenv()

# Verify Key
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("Enter OpenAI API Key: ")

print("Environment Ready.")

Environment Ready.


## **Load the data**

In [2]:
import json
import pandas as pd
import os

# Define Paths
catalog_path = "data/product_catalog.json"
faq_path = "data/faq.csv"
logic_path = "data/business_logic.csv"
policy_path = "data/policy.txt"

# 1. Load Product Catalog
if os.path.exists(catalog_path):
    with open(catalog_path, "r") as f:
        product_data = json.load(f)
    print(f"✅ Loaded Product Catalog: {len(product_data)} items")
else:
    print(f"❌ Error: {catalog_path} not found.")

# 2. Load FAQ (General)
if os.path.exists(faq_path):
    faq_data = pd.read_csv(faq_path)
    print(f"✅ Loaded FAQ: {len(faq_data)} rows")
else:
    print(f"❌ Error: {faq_path} not found.")

# 3. Load Business Logic (Deterministic)
if os.path.exists(logic_path):
    logic_data = pd.read_csv(logic_path)
    print(f"✅ Loaded Business Logic: {len(logic_data)} rows")
else:
    print(f"❌ Error: {logic_path} not found.")

# 4. Load Policy
if os.path.exists(policy_path):
    with open(policy_path, "r") as f:
        policy_text = f.read()
    print(f"✅ Loaded Policy: {len(policy_text)} chars")
else:
    print(f"❌ Error: {policy_path} not found.")

✅ Loaded Product Catalog: 5 items
✅ Loaded FAQ: 17 rows
✅ Loaded Business Logic: 9 rows
✅ Loaded Policy: 2449 chars


## **Process the data**

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_real_data_for_indexing():
    documents = []

    # --- 1. JSON: Product Catalog (HQI Strategy) ---
    # We embed the QUESTIONS, but return the CONTENT.
    for item in product_data:
        for q in item['indexing_questions']:
            doc = Document(
                page_content=q,  # <--- Embed the Question
                metadata={
                    "source": "product_catalog",
                    "answer_content": item['content'], # <--- LLM reads this
                    "url_context": item.get('url_context', '/'),
                    "category": item['category'],
                    "type": "technical"
                }
            )
            documents.append(doc)

    # --- 2. CSV: FAQ (Standard Match) ---
    for _, row in faq_data.iterrows():
        doc = Document(
            page_content=row['question'],
            metadata={
                "source": "faq",
                "answer_content": row['answer'],
                "category": row['category'],
                "type": "general_faq"
            }
        )
        documents.append(doc)

    # --- 3. CSV: Business Logic (High Priority Match) ---
    # We treat these same as FAQ but tag them 'business_logic'
    for _, row in logic_data.iterrows():
        doc = Document(
            page_content=row['question'],
            metadata={
                "source": "business_logic",
                "answer_content": row['answer'],
                "category": row['category'],
                "type": "deterministic"
            }
        )
        documents.append(doc)

    # --- 4. TXT: Policy (Chunking) ---
    # Policy is long text, so we chunk it using standard splitting.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    policy_chunks = text_splitter.create_documents([policy_text])
    
    for chunk in policy_chunks:
        chunk.metadata["source"] = "policy"
        chunk.metadata["type"] = "legal"
        # For policy, the answer is the chunk itself
        chunk.metadata["answer_content"] = chunk.page_content 
        documents.append(chunk)

    return documents

# Execute Processing
raw_docs = process_real_data_for_indexing()
print(f"🎉 Processed {len(raw_docs)} total vectors for indexing.")
print(f"Sample Metadata: {raw_docs[0].metadata}")

🎉 Processed 62 total vectors for indexing.
Sample Metadata: {'source': 'product_catalog', 'answer_content': 'BYV architects high-performance digital platforms using Next.js and Cloud-Native technologies. Unlike agencies that rely on brittle legacy systems (like WordPress), we build API-first architectures using React, FastAPI, and PostgreSQL. Key features include sub-100ms load times via Edge Caching, Enterprise Security (SOC-2 Ready, 256-bit encryption), and Mobile Native deployment (iOS/Android from a single codebase). This infrastructure is designed for scalability, automatically handling traffic spikes without downtime.', 'url_context': '/products/infrastructure', 'category': 'infrastructure', 'type': 'technical'}
